This is just a file that we use to verify that the methods function as expected.

In [65]:
import numpy as np 
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

In [43]:
### helper methods
# to go from and back to matrix indices
N = 3
def decode_index(idx):
    return idx % N, idx//N

def encode_index(x, y):
    return x + N*y

# construct the transition matrix
def transition_matrix(N, p):
    transition_matrix = np.zeros((N**2, N**2))
    transition_positions = []
    for node in range(N**2):
        x_1, y_1 = decode_index(node)
        x_2 = (x_1 + 1) % N
        y_2 = (y_1 + 1) % N

        right_node = encode_index(x_2, y_1)
        up_node = encode_index(x_1, y_2)

        transition_matrix[node][right_node] = (p)
        transition_matrix[node][up_node] = (1-p)
        transition_positions.append((node, right_node))
        transition_positions.append((node, up_node))
    
    print(transition_positions)
    return transition_matrix

In [80]:
def transition_positions(N, p):
    right_positions = []
    up_positions = []
    for node in range(N**2):
        x_1, y_1 = decode_index(node)
        x_2 = (x_1 + 1) % N
        y_2 = (y_1 + 1) % N

        right_node = encode_index(x_2, y_1)
        up_node = encode_index(x_1, y_2)

        right_positions.append((node, right_node))
        up_positions.append((node, up_node))

    # data vector
    right_data = np.array([p]*len(right_positions))
    up_data = np.array([1-p]*len(up_positions))
    # data = np.concatenate((right_ps, up_ps), axis=None) 

    # row vectors 
    print("positions are ", right_positions)
    right_rows = np.array([k[0] for k in right_positions])
    right_cols = np.array([k[1] for k in right_positions])

    up_rows = np.array([k[0] for k in up_positions])
    up_cols = np.array([k[1] for k in up_positions])
    print(right_data)
    print(right_rows)
    print(right_cols)
    csr_right = csr_matrix((right_data, (right_rows, right_cols)), shape=(N**2,N**2))
    csr_up = csr_matrix((up_data, (up_rows, up_cols)), shape=(N**2,N**2))

    ones = np.ones(N**2)
    csr_identity = csr_matrix((ones, (range(N**2), range(N**2))))

    generator_matrix = csr_identity - (csr_right + csr_up)

    return generator_matrix

def generator_computation(generator_matrix):
    u, s, vt = svds(generator_matrix, k=2, which="LM")

    return s


In [81]:
L = transition_positions(3, 0.5)
L.toarray()

positions are  [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3), (6, 7), (7, 8), (8, 6)]
[0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]
[0 1 2 3 4 5 6 7 8]
[1 2 0 4 5 3 7 8 6]


array([[ 1. , -0.5,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  1. , -0.5,  0. , -0.5,  0. ,  0. ,  0. ,  0. ],
       [-0.5,  0. ,  1. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  1. , -0.5,  0. , -0.5,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  1. , -0.5,  0. , -0.5,  0. ],
       [ 0. ,  0. ,  0. , -0.5,  0. ,  1. ,  0. ,  0. , -0.5],
       [-0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  1. , -0.5,  0. ],
       [ 0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  1. , -0.5],
       [ 0. ,  0. , -0.5,  0. ,  0. ,  0. , -0.5,  0. ,  1. ]])

In [89]:
u, s, vt = svds(L,k=8)
s

array([0.8660254 , 0.8660254 , 0.8660254 , 0.8660254 , 1.5       ,
       1.5       , 1.73205081, 1.73205081])

In [39]:
p = 0.5
I = np.eye(N**2)
P = transition_matrix(N, p)

In [40]:
I - P

array([[ 1. , -0.5,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  1. , -0.5,  0. , -0.5,  0. ,  0. ,  0. ,  0. ],
       [-0.5,  0. ,  1. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  1. , -0.5,  0. , -0.5,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  1. , -0.5,  0. , -0.5,  0. ],
       [ 0. ,  0. ,  0. , -0.5,  0. ,  1. ,  0. ,  0. , -0.5],
       [-0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  1. , -0.5,  0. ],
       [ 0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  1. , -0.5],
       [ 0. ,  0. , -0.5,  0. ,  0. ,  0. , -0.5,  0. ,  1. ]])

In [70]:
np.linalg.svdvals(I-P)

array([1.73205081e+00, 1.73205081e+00, 1.50000000e+00, 1.50000000e+00,
       8.66025404e-01, 8.66025404e-01, 8.66025404e-01, 8.66025404e-01,
       1.22828971e-16])

In [64]:
I-P == transition_matrix_two.toarray()

array([[ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True]])

In [41]:
np.linalg.svdvals(I - P)

array([1.73205081e+00, 1.73205081e+00, 1.50000000e+00, 1.50000000e+00,
       8.66025404e-01, 8.66025404e-01, 8.66025404e-01, 8.66025404e-01,
       1.22828971e-16])

In [44]:
P = transition_matrix(N, p)
# P[0][2] = P[0][2] + 1

P

[(0, 1), (0, 3), (1, 2), (1, 4), (2, 0), (2, 5), (3, 4), (3, 6), (4, 5), (4, 7), (5, 3), (5, 8), (6, 7), (6, 0), (7, 8), (7, 1), (8, 6), (8, 2)]


array([[0. , 0.5, 0. , 0.5, 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0.5, 0. , 0.5, 0. , 0. , 0. , 0. ],
       [0.5, 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0.5, 0. , 0.5, 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0.5, 0. ],
       [0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0.5],
       [0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. ],
       [0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5],
       [0. , 0. , 0.5, 0. , 0. , 0. , 0.5, 0. , 0. ]])